In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
import joblib
from sklearn.model_selection import train_test_split


DATA_DIR = "D:/Machine-Learning/HousePrice prediction/data"
x = pd.read_csv(f"{DATA_DIR}/x.csv")
y = pd.read_csv(f"{DATA_DIR}/y.csv").squeeze()  # Convert DataFrame to Series
results = []

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.23,
    random_state=42
)


In [2]:
def save_predictions(model_name, y_pred, y_test):
    prediction_df = pd.DataFrame({
        'Actual_Price': y_test.values,
        'Predicted_Price': y_pred,
    })

    prediction_df['Residual'] = (
        prediction_df['Actual_Price'] -
        prediction_df['Predicted_Price']
    )

    prediction_df['Absolute_Error'] = (
        abs(prediction_df['Residual'])
    )

    print(prediction_df.head())

    prediction_df.to_csv(
        "./models/predictions/" + model_name + "_predictions.csv",
        index=False
    )


# Linear Regression

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

pipeline = Pipeline([
    # feature scaling (normalizing)
    # rescales every features between 0 to 1  xScaled = (x-xmin)/(xmax-xmin)
    ('scaler', MinMaxScaler()),
    ('rfe', RFE(LinearRegression())),
    ('model', LinearRegression())
])

param_grid = {
    'rfe__n_features_to_select': [3,5,7,10,12]
}

# KFold CV splits data for cross-validation after model is trained
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# tries different parameter combinations automatically
grid = GridSearchCV(estimator=pipeline,
                     param_grid=param_grid, 
                     cv=kf, 
                     scoring='r2',
                     n_jobs=1

                     )
grid.fit(x_train, y_train)

print("Best R2 score:", grid.best_score_)
print("Best number of features:", grid.best_params_['rfe__n_features_to_select'])
best_pipeline = grid.best_estimator_

rfe = best_pipeline.named_steps['rfe']

selected_features = x.columns[rfe.support_]

print("Selected features:", selected_features.tolist())

y_pred = best_pipeline.predict(x_test)

# save to excel
save_predictions("linear_regression", y_pred, y_test)

# Selected features from best pipeline
X_selected = x_train.loc[:, selected_features]

# Add constant for statsmodels
X_selected_const = sm.add_constant(X_selected)

# Calculate VIF
vif = pd.DataFrame()
vif['Feature'] = X_selected_const.columns
vif['VIF'] = [variance_inflation_factor(X_selected_const.values, i) 
               for i in range(X_selected_const.shape[1])]

vif['VIF'] = round(vif['VIF'], 2)
vif = vif.sort_values(by='VIF', ascending=False)

print(vif)


print("R2:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
mean_price = y.mean()

rmse_percent = (1133040 / mean_price) * 100

print(rmse_percent)

joblib.dump(best_pipeline, "./models/linear_regression.pkl")
results.append({
    "Model": "Linear Regression",
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "R2": r2_score(y_test, y_pred),
    "MAE": mean_absolute_error(y_test, y_pred)
})


Best R2 score: 0.64201624518641
Best number of features: 12
Selected features: ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus_unfurnished']
   Actual_Price  Predicted_Price      Residual  Absolute_Error
0     4060000.0     5.076879e+06 -1.016879e+06    1.016879e+06
1     6650000.0     7.045816e+06 -3.958160e+05    3.958160e+05
2     3710000.0     3.232227e+06  4.777725e+05    4.777725e+05
3     6440000.0     4.710137e+06  1.729863e+06    1.729863e+06
4     2800000.0     3.288785e+06 -4.887847e+05    4.887847e+05
                         Feature    VIF
0                          const  30.07
4                        stories   1.47
1                           area   1.37
2                       bedrooms   1.37
7                       basement   1.32
3                      bathrooms   1.29
6                      guestroom   1.22
10                       parking   1.22
9       

# Decision Tree

In [4]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(random_state=42)

param_grid = {
    'max_depth': [2, 3, 4, 5, 10, 20, None],
    'min_samples_leaf': [1, 2, 4, 5, 10],
    'min_samples_split': [2, 5, 10]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    model,
    param_grid,
    cv=kf,
    scoring='r2',
    n_jobs=-1
)

grid.fit(x_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

print("\nBest CV R² Score:")
print(grid.best_score_)

print("\nBest Estimator:")
print(grid.best_estimator_)

grid.fit(x_train, y_train)

best_model = grid.best_estimator_

y_pred = best_model.predict(x_test)

save_predictions("decision_tree", y_pred, y_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

joblib.dump(best_model, "./models/decision_tree.pkl")
results.append({
    "Model": "Decision Tree",
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "R2": r2_score(y_test, y_pred),
    "MAE": mean_absolute_error(y_test, y_pred)
})


Best Parameters:
{'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 2}

Best CV R² Score:
0.5430636061419067

Best Estimator:
DecisionTreeRegressor(max_depth=5, min_samples_leaf=4, random_state=42)
   Actual_Price  Predicted_Price      Residual  Absolute_Error
0     4060000.0     4.875500e+06 -8.155000e+05    8.155000e+05
1     6650000.0     7.249200e+06 -5.992000e+05    5.992000e+05
2     3710000.0     4.127631e+06 -4.176308e+05    4.176308e+05
3     6440000.0     4.127631e+06  2.312369e+06    2.312369e+06
4     2800000.0     4.127631e+06 -1.327631e+06    1.327631e+06
MAE: 1109590.8108802966
RMSE: 1429795.4467559077
R2: 0.480157005700278


# Random Forest

In [5]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=42)

# v imp as it can improve the model
param_grid = {
    'n_estimators': [50, 100, 150],  
    'max_depth': [None, 10, 20], 
    'min_samples_split': [2, 5, 10],  
    'min_samples_leaf': [1, 2, 4], 
    'max_features': ['sqrt', 'log2', None]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    model,
    param_grid,
    cv=kf,
    scoring='r2',
    n_jobs=-1
)

grid.fit(x_train, y_train)
best_model = grid.best_estimator_

print(best_model)
y_pred = best_model.predict(x_test)

save_predictions("random_forest", y_pred, y_test)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

joblib.dump(best_model, "./models/random_forest.pkl")
results.append({
    "Model": "Random Forest",
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "R2": r2_score(y_test, y_pred),
    "MAE": mean_absolute_error(y_test, y_pred)
})


RandomForestRegressor(max_depth=10, max_features='sqrt', min_samples_split=5,
                      n_estimators=150, random_state=42)
   Actual_Price  Predicted_Price      Residual  Absolute_Error
0     4060000.0     5.309622e+06 -1.249622e+06    1.249622e+06
1     6650000.0     7.153994e+06 -5.039937e+05    5.039937e+05
2     3710000.0     3.729121e+06 -1.912065e+04    1.912065e+04
3     6440000.0     4.670053e+06  1.769947e+06    1.769947e+06
4     2800000.0     3.933979e+06 -1.133979e+06    1.133979e+06
MAE: 884789.5233577655
RMSE: 1138949.5238341957
R2: 0.6701372077760688


# GradientBoostModel

In [6]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, KFold

model = GradientBoostingRegressor(random_state=42)

param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=kf,
    scoring='r2',
    n_jobs=-1
)

grid.fit(x_train, y_train)

best_model = grid.best_estimator_

print("Best Parameters:")
print(grid.best_params_)

print("\nBest CV R2:")
print(grid.best_score_)

y_pred = best_model.predict(x_test)

save_predictions("gradient_boosting", y_pred, y_test)
print("\nTest R2:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE:", mean_absolute_error(y_test, y_pred))

joblib.dump(best_model, "./models/gradient_boosting.pkl")
results.append({
    "Model": "Gradient Boosting",
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "R2": r2_score(y_test, y_pred),
    "MAE": mean_absolute_error(y_test, y_pred)
})


Best Parameters:
{'learning_rate': 0.01, 'max_depth': 4, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 500}

Best CV R2:
0.6500939718843183
   Actual_Price  Predicted_Price      Residual  Absolute_Error
0     4060000.0     4.498324e+06 -4.383235e+05    4.383235e+05
1     6650000.0     7.314621e+06 -6.646213e+05    6.646213e+05
2     3710000.0     3.663908e+06  4.609195e+04    4.609195e+04
3     6440000.0     4.418481e+06  2.021519e+06    2.021519e+06
4     2800000.0     3.940589e+06 -1.140589e+06    1.140589e+06

Test R2: 0.670560509229619
RMSE: 1138218.5020710798
MAE: 879303.4683014815


# XGBoost Model

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

model = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
       tree_method='hist'
)

param_grid = {
    'n_estimators': [100, 200, 300, 500, 700],

    'learning_rate': [0.005, 0.01, 0.03, 0.05, 0.1],

    'max_depth': [2, 3, 4, 5, 6, 8],

    'subsample': [0.6, 0.7, 0.8, 1.0],

    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],

    'min_child_weight': [1, 3, 5, 7],

    'gamma': [0, 0.1, 0.3, 0.5],

    'reg_alpha': [0, 0.01, 0.1, 1],

    'reg_lambda': [0.1, 1, 5, 10]
}

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=75,
    scoring='r2',
    cv=kf,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(
    x_train,
    y_train,
    eval_set=[(x_test, y_test)],
    early_stopping_rounds=50,
    verbose=False
)

best_model = random_search.best_estimator_

print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV R2:")
print(random_search.best_score_)

y_pred = best_model.predict(x_test)

save_predictions("xgboost", y_pred, y_test)
mse = mean_squared_error(y_test, y_pred)

print("\nTest R2:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mse))
print("MAE:", mean_absolute_error(y_test, y_pred))

joblib.dump(best_model, "./models/xgboost.pkl")
results.append({
    "Model": "XGBoost",
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "R2": r2_score(y_test, y_pred),
    "MAE": mean_absolute_error(y_test, y_pred)
})


Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best Parameters:
{'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.01, 'n_estimators': 700, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.005, 'gamma': 0.3, 'colsample_bytree': 0.7}

Best CV R2:
0.6521475370106671
   Actual_Price  Predicted_Price    Residual  Absolute_Error
0     4060000.0       5195041.50 -1135041.50      1135041.50
1     6650000.0       6951643.00  -301643.00       301643.00
2     3710000.0       3687360.75    22639.25        22639.25
3     6440000.0       4506910.00  1933090.00      1933090.00
4     2800000.0       3981159.00 -1181159.00      1181159.00

Test R2: 0.6709074891299951
RMSE: 1137618.9337132922
MAE: 867479.9583333334


In [8]:
#save results to csv
results_df = pd.DataFrame(results)

print(results_df)
results_df.to_csv("./models/model_results.csv", index=False)


               Model          RMSE        R2           MAE
0  Linear Regression  1.083668e+06  0.701381  8.355465e+05
1      Decision Tree  1.429795e+06  0.480157  1.109591e+06
2      Random Forest  1.138950e+06  0.670137  8.847895e+05
3  Gradient Boosting  1.138219e+06  0.670561  8.793035e+05
4            XGBoost  1.137619e+06  0.670907  8.674800e+05
